# 21 — Pull ICEWS Monthly Aggregates (Harvard Dataverse)

**Source:** Integrated Crisis Early Warning System (ICEWS)  
**Provider:** Lockheed Martin / Raytheon BBN → released via Harvard Dataverse  
**DOI:** [10.7910/DVN/28075](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/28075)  
**Access:** Harvard Dataverse API — free, no registration  
**Coverage:** Global, 1995–present  
**Frequency:** Aggregated here to country-month  

## What this notebook does
ICEWS provides CAMEO-coded political events extracted from news, processed by a
government-grade NLP pipeline distinct from GDELT. Using both gives two partially
independent media-derived signals from different processing chains.

We do **not** store raw event rows (tens of millions per year). We download each
annual file, aggregate to country-month in memory, and discard the raw rows.

## Features produced

| Feature | Description |
|---|---|
| `goldstein_mean` | Monthly mean Goldstein Scale (−10 to +10) |
| `goldstein_std` | Monthly volatility |
| `events_total` | Total event count |
| `events_conflict` | Material/verbal conflict events (QuadClass 3+4) |
| `events_coop` | Cooperative events (QuadClass 1+2) |
| `conflict_coop_ratio` | events_conflict / (events_total + 1) |

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

## Note on size
Each annual compressed file is 50–150 MB. This notebook processes one year at a time,
aggregates in memory, then releases the raw DataFrame before proceeding to the next year.

In [ ]:
import os
import io
import time
import zipfile
import requests
import pandas as pd
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

START_YEAR = 2000
END_YEAR   = datetime.utcnow().year - 1  # current year file may not yet be published

# Harvard Dataverse API — list files in ICEWS dataset
DATAVERSE_BASE   = "https://dataverse.harvard.edu/api"
ICEWS_PERSISTENT = "doi:10.7910/DVN/28075"

print(f"Year range  : {START_YEAR}–{END_YEAR}")
print(f"ADLS target : abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/raw/icews/")
print(f"Run date    : {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Enumerate ICEWS files on Harvard Dataverse

The Dataverse API returns a manifest of all files in the dataset. We filter to the
annual `.tab.zip` event files and build a year → download URL map.

In [ ]:
resp = requests.get(
    f"{DATAVERSE_BASE}/datasets/:persistentId/versions/:latest/files",
    params={"persistentId": ICEWS_PERSISTENT},
    timeout=30,
)
resp.raise_for_status()
file_list = resp.json()["data"]

# Each entry has: dataFile.id, dataFile.filename, dataFile.filesize
year_to_fileid = {}
for entry in file_list:
    fname = entry["dataFile"]["filename"]
    # Filename pattern: events.YYYY.YYYYMMDD.tab.zip
    if fname.startswith("events.") and fname.endswith(".tab.zip"):
        parts = fname.split(".")
        if len(parts) >= 2 and parts[1].isdigit():
            yr = int(parts[1])
            if START_YEAR <= yr <= END_YEAR:
                year_to_fileid[yr] = entry["dataFile"]["id"]

print(f"Found {len(year_to_fileid)} annual files covering {sorted(year_to_fileid.keys())[0]}–{sorted(year_to_fileid.keys())[-1]}")

## Aggregate ICEWS year-by-year

For each year: download the zip, decompress into memory, read the TSV,
aggregate to country-month, then drop the raw DataFrame to free memory.

In [ ]:
ICEWS_COLS = {
    "Event Date":      "event_date",
    "Source Country":  "source_country",
    "Country":         "country",
    "Intensity":       "goldstein",   # Goldstein scale in ICEWS
    "CAMEO Code":      "cameo_code",
    "Quad Category":   "quad_class",  # 1=verbal coop, 2=material coop, 3=verbal conf, 4=material conf
}

monthly_frames = []

for yr in sorted(year_to_fileid.keys()):
    file_id = year_to_fileid[yr]
    url = f"{DATAVERSE_BASE}/access/datafile/{file_id}"
    print(f"  {yr}: downloading...", end=" ", flush=True)

    for attempt in range(1, 4):
        try:
            r = requests.get(url, timeout=300, stream=True)
            r.raise_for_status()
            break
        except Exception as exc:
            if attempt == 3:
                print(f"FAILED ({exc}) — skipping {yr}")
                continue
            time.sleep(10 * attempt)

    # Decompress in memory
    zf = zipfile.ZipFile(io.BytesIO(r.content))
    tab_name = [n for n in zf.namelist() if n.endswith(".tab")][0]
    df_raw = pd.read_csv(zf.open(tab_name), sep="\t", low_memory=False,
                         usecols=lambda c: c in ICEWS_COLS)
    df_raw.rename(columns=ICEWS_COLS, inplace=True)
    print(f"{len(df_raw):,} events", end=" → ", flush=True)

    # Parse date and derive year_month
    df_raw["event_date"] = pd.to_datetime(df_raw["event_date"], errors="coerce")
    df_raw["year_month"] = df_raw["event_date"].dt.to_period("M").astype(str)
    df_raw["goldstein"]  = pd.to_numeric(df_raw["goldstein"], errors="coerce")
    df_raw["quad_class"] = pd.to_numeric(df_raw["quad_class"], errors="coerce")
    df_raw["is_conflict"]= df_raw["quad_class"].isin([3, 4]).astype(int)
    df_raw["is_coop"]    = df_raw["quad_class"].isin([1, 2]).astype(int)

    agg = (
        df_raw
        .groupby(["country", "year_month"], as_index=False)
        .agg(
            events_total   =("goldstein", "count"),
            goldstein_mean =("goldstein", "mean"),
            goldstein_std  =("goldstein", "std"),
            events_conflict=("is_conflict", "sum"),
            events_coop    =("is_coop", "sum"),
        )
    )
    agg["conflict_coop_ratio"] = agg["events_conflict"] / (agg["events_total"] + 1)
    monthly_frames.append(agg)
    print(f"{len(agg):,} country-months")

    # Free raw memory before next year
    del df_raw

df_icews = pd.concat(monthly_frames, ignore_index=True)
df_icews["goldstein_std"] = df_icews["goldstein_std"].fillna(0)
print(f"\nFinal shape: {df_icews.shape}")
df_icews.head()

## Write to ADLS

In [ ]:
write_parquet(df_icews, f"raw/icews/{RUN_DATE}/icews_monthly.parquet")

## Summary

In [ ]:
print("=" * 55)
print("ICEWS pull complete")
print("=" * 55)
print(f"  Country-months : {len(df_icews):,}")
print(f"  Countries      : {df_icews['country'].nunique()}")
year_range = df_icews["year_month"].agg(["min", "max"])
print(f"  Year-month     : {year_range['min']} → {year_range['max']}")
print()
print("ADLS path written:")
print(f"  raw/icews/{RUN_DATE}/icews_monthly.parquet")